# 06 - Synthetic readout data: generate it, and see what it misses

Can a classifier be trained on synthetic readout traces instead of recorded ones? Paper Sec. V-C answers this with a
generator fit to the recorded data. This notebook builds that generator, checks it against the real data, and shows the
two differences that matter.

**The generator.** Each synthetic shot of state $s$ is

$$z = m_s\, e^{i\phi} + n, \qquad \phi \sim \mathcal N(0,\, 0.1\ \text{rad}), \qquad n \sim \mathcal N(0, \Sigma_s),$$

where $m_s$ is the class-mean trajectory and $\Sigma_s$ the full $1540 \times 1540$ covariance of the I/Q noise around it,
both measured on recorded shots. A full covariance carries the noise's colour (correlation from one sample to the next)
and the per-shot offset that does not average down. A model with independent noise per sample makes the data easier
than it is.

**What it cannot reproduce:** anything that is not Gaussian around the class mean. Real |1⟩ shots that decay during
the window, and |0⟩ shots that start excited, pull the two classes toward each other. The generator makes symmetric
classes instead.

**You need:** the dataset's training split (`0528_X_train_0_770.npy`, `0528_y_train_0_770.npy`). The fit uses rows
10,000–110,000; the deployed networks never train on them.

In [ ]:
from pathlib import Path

DATA_DIR = Path("..")
OUT_DIR = Path("outputs/synthetic")
FIT_ROWS = (10000, 110000)    # generator fit pool (kept out of every classifier's training set)
COV_SHOTS = 40000             # shots per class used for the covariance
N_SHOTS = 50000               # synthetic shots to generate (the paper's set)
SEED = 20260902               # the paper's synthetic set used this seed
PHASE_SIGMA = 0.1             # rad, per-shot phase jitter
N_COMPARE = 6000              # real and synthetic shots per class for the comparisons

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import skew

X = np.load(DATA_DIR / "0528_X_train_0_770.npy", mmap_mode="r")
y = np.load(DATA_DIR / "0528_y_train_0_770.npy").astype(int)
Xp = np.asarray(X[FIT_ROWS[0]:FIT_ROWS[1]]).astype(np.float64)
yp = y[FIT_ROWS[0]:FIT_ROWS[1]]
X0, X1 = Xp[yp == 0], Xp[yp == 1]
print(f"fit pool: {len(X0)} |0> shots, {len(X1)} |1> shots, {Xp.shape[1]} values each (interleaved I, Q)")

## 1. Fit: class means and noise covariances

The covariance is real-valued over the interleaved I and Q values, so it also holds any I/Q correlation. A tiny ridge
keeps the Cholesky factorization stable.

In [ ]:
def chol_cov(noise):
    Cm = np.cov(noise, rowvar=False)
    Cm = 0.5 * (Cm + Cm.T) + np.eye(Cm.shape[0]) * 1e-9 * np.trace(Cm) / Cm.shape[0]
    try:
        return np.linalg.cholesky(Cm)
    except np.linalg.LinAlgError:
        w, V = np.linalg.eigh(Cm)
        return V @ np.diag(np.sqrt(np.clip(w, 0, None)))


mean_iq = [X0.mean(axis=0), X1.mean(axis=0)]
L = [chol_cov(X0[:COV_SHOTS] - mean_iq[0]), chol_cov(X1[:COV_SHOTS] - mean_iq[1])]
m = [mi[0::2] + 1j * mi[1::2] for mi in mean_iq]

noise0 = (X0[:COV_SHOTS, 0::2] + 1j * X0[:COV_SHOTS, 1::2]) - m[0]
ac = [np.mean(np.real(noise0[:, :-k] * np.conj(noise0[:, k:]))) / np.mean(np.abs(noise0) ** 2) for k in range(1, 6)]
print("noise autocorrelation at lags 1-5 (|0>):", np.round(ac, 3))
print(f"per-sample noise std: {np.std(noise0):.0f}; |m1 - m0| averaged over the trace: {np.mean(np.abs(m[1] - m[0])):.0f}")

## 2. Generate

The state is a fair coin per shot. The draw order below (state, then phase, then noise, shot by shot) reproduces the
paper's synthetic set exactly for `SEED = 20260902`.

In [ ]:
def generate(n_shots, seed, phase_sigma=PHASE_SIGMA):
    states = np.random.RandomState(seed).randint(0, 2, size=n_shots)
    rng = np.random.default_rng(seed)
    Xg = np.empty((n_shots, 2 * len(m[0])))
    for i, s in enumerate(states):
        phase = rng.normal(0, phase_sigma)
        noise = L[s] @ rng.normal(size=L[s].shape[0])
        z = m[s] * np.exp(1j * phase) + (noise[0::2] + 1j * noise[1::2])
        Xg[i, 0::2], Xg[i, 1::2] = z.real, z.imag
    return Xg, states


Xg, yg = generate(N_SHOTS, SEED)
np.save(OUT_DIR / "x_synth.npy", Xg); np.save(OUT_DIR / "y_synth.npy", yg)
print(f"generated {Xg.shape}, balance {np.bincount(yg)}")

## 3. Acceptance test: the matched-filter ceiling

The best linear discriminant for Gaussian noise projects a trace onto $w = m_1 - m_0$. Fit $w$ on half the shots, score
the other half. It reports the separation $d$ (difference of class means over the pooled standard deviation of the
scores) and the fidelity. A generator whose ceiling sits above the real data's is making the problem easier than it is.

In [ ]:
def mf_scores(Xa, ya, seed=0):
    z = Xa[:, 0::2] + 1j * Xa[:, 1::2]
    rng = np.random.default_rng(seed)
    i0, i1 = rng.permutation(np.flatnonzero(ya == 0)), rng.permutation(np.flatnonzero(ya == 1))
    h0, h1 = len(i0) // 2, len(i1) // 2
    w = z[i1[:h1]].mean(0) - z[i0[:h0]].mean(0)
    s0, s1 = (z[i0[h0:]] * np.conj(w)).real.sum(1), (z[i1[h1:]] * np.conj(w)).real.sum(1)
    return s0, s1


def ceiling(Xa, ya):
    s0, s1 = mf_scores(Xa, ya)
    d = abs(s1.mean() - s0.mean()) / np.sqrt(0.5 * (s0.var() + s1.var()))
    thr = 0.5 * (s0.mean() + s1.mean())
    return d, 0.5 * (np.mean(s0 < thr) + np.mean(s1 > thr))


n = N_COMPARE
Xr = np.concatenate([X0[:n], X1[:n]]); yr = np.r_[np.zeros(n, int), np.ones(n, int)]
Xs = np.concatenate([Xg[yg == 0][:n], Xg[yg == 1][:n]]); ys = yr.copy()
for name, Xa, ya in (("real", Xr, yr), ("synthetic", Xs, ys)):
    d, F = ceiling(Xa, ya)
    print(f"{name:9s}: d = {d:.3f} sigma, matched-filter fidelity {100 * F:.2f}%")

## 4. What the generator misses

**(a) Score distributions.** Same separation, but the real classes lean toward each other: |1⟩ has a tail toward |0⟩
(decay during the window) and |0⟩ a tail toward |1⟩ (thermal population, preparation error). The synthetic classes are
symmetric.

**(b) Window length.** On real data the matched filter peaks at an interior window, because later samples carry more
decay than signal. Synthetic data has no decay, so its fidelity keeps rising to the end of the trace.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, Xa, ya, ls in (("real", Xr, yr, "-"), ("synthetic", Xs, ys, "--")):
    s0, s1 = mf_scores(Xa, ya)
    mu0, mu1 = s0.mean(), s1.mean()
    sd = np.sqrt(0.5 * (s0.var() + s1.var()))
    sign = 1 if mu1 > mu0 else -1
    z0, z1 = sign * (s0 - mu0) / sd, sign * (s1 - mu0) / sd
    bins = np.linspace(-4, 8, 120)
    ax[0].hist(z0, bins, histtype="step", ls=ls, color="C0", density=True, label=f"{name} |0⟩")
    ax[0].hist(z1, bins, histtype="step", ls=ls, color="C3", density=True, label=f"{name} |1⟩")
    print(f"{name:9s}: score skewness |0> {sign * skew(s0):+.2f}, |1> {sign * skew(s1):+.2f}")
ax[0].set_xlabel("matched-filter score (standardized)"); ax[0].set_yscale("log"); ax[0].legend(fontsize=7)

windows = np.arange(50, 771, 30)
for name, Xa, ya, ls in (("real", Xr, yr, "-"), ("synthetic", Xs, ys, "--")):
    F = [ceiling(Xa[:, :2 * w], ya)[1] for w in windows]
    ax[1].plot(windows, 100 * np.array(F), ls, label=name)
ax[1].set_xlabel("window length (samples from the start; 3.25 ns each in the recording)"); ax[1].set_ylabel("matched-filter fidelity (%)")
ax[1].legend(); plt.tight_layout(); plt.show()

**What this means for training** (paper Sec. V-C, Fig. 3(c)): a network trained only on synthetic data has never seen
the skewed tails, so it does worse on real replayed traces than one trained on real data.

**Things to try:** set the phase jitter to 0, replace the covariance with its diagonal (independent noise per sample)
and watch the ceiling rise, or fit the generator on a few hundred shots instead of 100,000 to see how much data the
covariance needs.

---
## Optional: the cavity-response model of `generate_data_4.ipynb`

Our first generator (`generate_data_4.ipynb`) built each trace from a fitted model of the readout resonator rather than
from the measured class means. The generator above does not use it, but the fit is a useful physical description of the
data, so it is kept here with the original's problems fixed (table below).

**The model.** The resonator field relaxes from its initial value $a_0$ to a state-dependent steady state $a_{ss,s}$,

$$a_s(t) = a_{ss,s} + (a_0 - a_{ss,s})\, e^{(-\kappa/2 + i\delta_s)\, t}, \qquad z_s(t) = g\, a_s(t)\, e^{i\theta} + (I_0 + i Q_0),$$

with linewidth $\kappa$, detunings $\delta_0, \delta_1$, gain $g$, phase $\theta$ and a constant offset $(I_0, Q_0)$ taken
from the last 100 samples. Both classes are fit jointly by weighted least squares, with weights $e^{-n/200}$ that
emphasize the ring-up.

**Fixes relative to `generate_data_4.ipynb`:**

| # | Original | Here |
|---|---|---|
| 1 | `generate_shot` used an undefined loop variable `t`, so only one of the 770 samples got noise | noise at every sample, drawn per sample from that sample's covariance |
| 2 | the per-sample noise covariance was measured on \|0⟩ only and reused for \|1⟩ | a separate covariance for each class |
| 3 | the noise *replaced* the signal, so the per-shot rotation was lost and the saved phases (`Phi`) described a rotation the traces did not have | signal rotated by the shot's phase, plus noise |
| 4 | the shot phase came from the global `np.random`, which the seed did not control | one seeded generator for everything |
| 5 | the fit pool (the full 1,000,000 shots) contained all 20,000 real test shots | fit on training rows 10,000–110,000, disjoint from the test file |
| 6 | the model's time step was 2 ns, but the recording's samples are 3.25 ns apart, so κ and δ came out in the wrong units | time step 3.25 ns, so κ and δ are physical |

The original's per-shot phases were a record only (`Phi.npy`), not a hardware input, and QICK's `phase` argument is in
degrees, not radians; none of the replay code sends them.

In [ ]:
from scipy.optimize import least_squares

DT = 3.25e-9                     # recording's sample spacing (fix 6; the original used 2e-9)
n = len(m[0])
offset = complex(np.mean(np.r_[m[0].real[-100:], m[1].real[-100:]]), np.mean(np.r_[m[0].imag[-100:], m[1].imag[-100:]]))


def cavity(params, state, n=n, dt=DT):
    kappa, d0, d1, s0r, s0i, s1r, s1i, a0r, a0i, theta, gain = params
    t = np.arange(n) * dt
    ass = (s0r + 1j * s0i) if state == 0 else (s1r + 1j * s1i)
    a = ass + ((a0r + 1j * a0i) - ass) * np.exp((-kappa / 2 + 1j * (d0 if state == 0 else d1)) * t)
    return gain * a * np.exp(1j * theta) + offset


def residual(params):
    w = np.exp(-np.arange(n) / 200)
    r0, r1 = w * (cavity(params, 0) - m[0]), w * (cavity(params, 1) - m[1])
    return np.concatenate([r0.real, r0.imag, r1.real, r1.imag])


# Starting point: the original's fitted values, with kappa and the detunings converted from its 2 ns step to 3.25 ns.
warm = np.array([1.01720706e+07, -4.54101244e+04, 3.64588334e+06, -3.07607069e-02, 2.98501165e-02, 2.70625462e-02,
                 -4.07594194e-02, 9.14256911e-02, 3.18694928e-02, 6.96860703e-06, 3.61911904e+03])
warm[:3] *= 2e-9 / DT
lower = [1e3] + [-np.inf] * 8 + [-np.pi, 0]
upper = [np.inf] * 9 + [np.pi, np.inf]
fit = least_squares(residual, warm, bounds=(lower, upper))
P = fit.x
kappa, d0, d1 = P[:3]
print(f"fit status {fit.status}; kappa/2pi = {kappa / 2 / np.pi / 1e6:.3f} MHz, "
      f"delta0/2pi = {d0 / 2 / np.pi / 1e6:+.3f} MHz, delta1/2pi = {d1 / 2 / np.pi / 1e6:+.3f} MHz, "
      f"|delta1 - delta0|/2pi = {abs(d1 - d0) / 2 / np.pi / 1e6:.3f} MHz")
for s_ in (0, 1):
    print(f"|{s_}>: mean |residual| {np.mean(np.abs(cavity(P, s_) - m[s_])):.1f} vs signal {np.mean(np.abs(m[s_] - offset)):.1f}")
sep_fit, sep_emp = np.mean(np.abs(cavity(P, 1) - cavity(P, 0))), np.mean(np.abs(m[1] - m[0]))
print(f"class separation |m1 - m0|: fit {sep_fit:.1f}, measured {sep_emp:.1f} ({100 * (sep_fit / sep_emp - 1):+.0f}%)")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
tt = np.arange(n) * DT * 1e6
for s_, col in ((0, "C0"), (1, "C3")):
    ax[0].plot(tt, np.abs(m[s_] - offset), col, label=f"measured |{s_}⟩")
    ax[0].plot(tt, np.abs(cavity(P, s_) - offset), col, ls="--", label=f"fit |{s_}⟩")
    ax[1].plot(m[s_].real, m[s_].imag, col); ax[1].plot(cavity(P, s_).real, cavity(P, s_).imag, col, ls="--")
ax[0].set_xlabel("time (us)"); ax[0].set_ylabel("|z - offset|"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("I"); ax[1].set_ylabel("Q"); ax[1].set_title("trajectories (dashed: fit)"); ax[1].axis("equal")
plt.tight_layout(); plt.show()

The paper describes the source device's readout as $\kappa/2\pi$ = 1.5 MHz and $2\chi/2\pi$ = 0.4 MHz. In this model the
dispersive shift appears as the difference of the two detunings. How closely the fit lands on those values shows how
far the simple single-mode model gets. The fit's class separation is also why the generator uses the measured means: an
error there changes how hard the synthetic data is.

**The original generator, corrected.** Cavity trajectory rotated by the shot's phase, plus independent noise at every
sample from that class's 2×2 I/Q covariance (fixes 1–4). Its matched-filter ceiling shows why it was replaced: white noise
ignores the sample-to-sample correlation and the per-shot offset, so it makes the problem much easier than the real data.

In [ ]:
def chol2(noise):                         # per-sample 2x2 I/Q covariance -> Cholesky factors, shape (n, 2, 2)
    out = np.empty((noise.shape[1], 2, 2))
    for t_ in range(noise.shape[1]):
        cv = np.cov(np.c_[noise[:, t_].real, noise[:, t_].imag], rowvar=False)
        out[t_] = np.linalg.cholesky(cv + np.eye(2) * 1e-6 * np.trace(cv))
    return out


Lw = [chol2(X0[:, 0::2] + 1j * X0[:, 1::2] - m[0]), chol2(X1[:, 0::2] + 1j * X1[:, 1::2] - m[1])]   # fix 2: one per class


def generate_white(n_shots, seed):
    rng = np.random.default_rng(seed)                                          # fix 4: one seeded generator
    states = rng.integers(0, 2, size=n_shots)
    Xw = np.empty((n_shots, 2 * n))
    for i, s_ in enumerate(states):
        z = cavity(P, s_) * np.exp(1j * rng.normal(0, PHASE_SIGMA))           # fix 3: rotate the signal ...
        e = np.einsum("tij,tj->ti", Lw[s_], rng.normal(size=(n, 2)))          # fix 1: noise at every sample
        z = z + (e[:, 0] + 1j * e[:, 1])                                       # ... then add the noise
        Xw[i, 0::2], Xw[i, 1::2] = z.real, z.imag
    return Xw, states


Xw, yw = generate_white(2 * N_COMPARE, SEED)
for name, Xa, ya in (("real", Xr, yr), ("generator above", Xs, ys), ("cavity + white noise", Xw, yw)):
    d, F = ceiling(Xa, ya)
    print(f"{name:21s}: d = {d:.3f} sigma, matched-filter fidelity {100 * F:.2f}%")